# Notebook 06 — Autoencoders and VAEs

**Module:** 14 — Deep Learning and GNNs  
**Tier:** 2 — Working competence  
**Notebook:** 6 of 12  
**Time estimate:** 90 minutes

> The variational autoencoder is the generative model behind scVI — the standard
> deep learning method for single-cell RNA-seq. Understanding the ELBO objective
> and the reparameterization trick is the prerequisite for reading any scRNA-seq
> deep learning paper published after 2018.

---
## Step 1 — Motivation

PCA finds the best *linear* low-dimensional representation. But biological data
lives on non-linear manifolds — cell types form curved clusters, not hyperplanes.
Autoencoders learn non-linear compressions. VAEs go further: they learn a
*probabilistic* latent space, enabling generation of new samples and
principled uncertainty quantification — critical for single-cell data where
counts are overdispersed and batches differ technically.

---
## Step 2 — Intuition

**Deterministic autoencoder:**
- Encoder $f_\phi$: compress $x$ → $z$ (bottleneck latent code)
- Decoder $g_\theta$: reconstruct $\hat{x}$ from $z$
- Train to minimize reconstruction loss: $\|x - \hat{x}\|^2$ (MSE) or cross-entropy
- Problem: latent space is irregular — points near $z$ may not decode sensibly

**Variational autoencoder (VAE):**
- Encoder outputs a distribution: $q_\phi(z|x) = \mathcal{N}(\mu(x), \text{diag}(\sigma^2(x)))$
- Decoder maps $z \sim q_\phi(z|x)$ → reconstruction
- Regularizer: push $q_\phi(z|x)$ toward the prior $p(z) = \mathcal{N}(0, I)$
- Result: a smooth, regular latent space where interpolation is meaningful

**Reparameterization trick:**
Cannot backpropagate through a sample $z \sim \mathcal{N}(\mu, \sigma^2)$.
Solution: sample $\epsilon \sim \mathcal{N}(0, I)$, then compute $z = \mu + \sigma \odot \epsilon$.
Gradient flows through $\mu$ and $\sigma$ (deterministic operations);
the randomness in $\epsilon$ is a constant w.r.t. parameters.

---
## Step 3 — Biological Background

**scVI (Lopez 2018, Nature Methods):**
- VAE for scRNA-seq count data
- Likelihood: negative binomial (captures count overdispersion)
- Batch variable $s$ injected into both encoder and decoder → learns batch-invariant
  biological variation in the latent space
- 10× faster than MCMC; latent space used for clustering, trajectory inference, DE
- 10 million cell training in minutes on GPU

**scANVI (Xu 2021):**
Extension of scVI that incorporates partial cell type labels (semi-supervised VAE).
Achieves better cell type separation than unsupervised VAE.

**VAE for protein design (Riesselman 2018):**
VAE trained on multiple sequence alignments (MSAs) of protein families.
Latent space encodes evolutionary constraints. Sampling from the prior generates
novel sequences with high probability of being functional.

**Latent space biology:**
In a well-trained scVI latent space:
- Cells of the same type cluster together
- Trajectory (pseudotime) can be computed by projecting onto the primary axis
- Differential expression: compare two populations' latent distributions
  and trace back which genes were responsible through the decoder

---
## Step 4 — Mathematical Explanation

**VAE objective (ELBO — Evidence Lower BOund):**
We want to maximize $\log p_\theta(x)$ (the marginal likelihood).
This is intractable (integral over all $z$). The ELBO is a lower bound:
$$\mathcal{L}_{ELBO}(\theta, \phi; x) = \underbrace{\mathbb{E}_{q_\phi(z|x)}[\log p_\theta(x|z)]}_{\text{reconstruction}} - \underbrace{D_{KL}(q_\phi(z|x) \| p(z))}_{\text{regularization}}$$

**KL divergence for Gaussian encoder vs. standard normal prior:**
$$D_{KL}(\mathcal{N}(\mu, \sigma^2 I) \| \mathcal{N}(0, I)) = \frac{1}{2}\sum_j (\mu_j^2 + \sigma_j^2 - \log\sigma_j^2 - 1)$$

This has a closed form — no Monte Carlo needed for the regularization term.

**Reparameterization trick:**
$$z = \mu_\phi(x) + \sigma_\phi(x) \odot \epsilon, \quad \epsilon \sim \mathcal{N}(0, I)$$

Gradient $\nabla_\phi \mathbb{E}_{q_\phi}[\cdot]$ is now computable via standard backprop.

**In practice:** encoder outputs $\mu$ and $\log\sigma^2$ (log-variance for numerical stability).

In [ ]:
# Step 6 — Python: VAE for simulated scRNA-seq data
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42); np.random.seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# ---- Simulated scRNA-seq data ----
rng = np.random.default_rng(42)
n_cells = 600
n_genes = 200
n_celltypes = 4

# Cell-type specific gene programs
gene_means = np.zeros((n_celltypes, n_genes))
for c in range(n_celltypes):
    gene_means[c, c*30:(c+1)*30] = rng.uniform(2, 5, 30)
    gene_means[c, 80+c*10:80+c*10+5] = rng.uniform(1, 3, 5)

cell_types = np.repeat(np.arange(n_celltypes), n_cells//n_celltypes)
# Negative binomial counts (approximated via Poisson with varying rate)
X_counts = np.zeros((n_cells, n_genes), dtype=np.float32)
for i in range(n_cells):
    c = cell_types[i]
    mean_expr = np.exp(rng.normal(gene_means[c], 0.4))
    X_counts[i] = rng.negative_binomial(2, 2/(2+mean_expr), n_genes).astype(float)

# Log-normalize (library size normalization)
lib_size = X_counts.sum(axis=1, keepdims=True)
X_lognorm = np.log1p(X_counts / (lib_size + 1) * 1e4).astype(np.float32)

print(f'scRNA-seq data: {n_cells} cells × {n_genes} genes')
print(f'Cell types: {n_celltypes}, cells per type: {n_cells//n_celltypes}')
print(f'Sparsity: {(X_counts==0).mean():.1%}')

# ---- VAE model ----
class VAE(nn.Module):
    def __init__(self, input_dim, latent_dim=10, hidden_dim=128):
        super().__init__()
        # Encoder
        self.enc_fc1 = nn.Linear(input_dim, hidden_dim)
        self.enc_fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc_mu = nn.Linear(hidden_dim, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim, latent_dim)
        # Decoder
        self.dec_fc1 = nn.Linear(latent_dim, hidden_dim)
        self.dec_fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.dec_out = nn.Linear(hidden_dim, input_dim)
    
    def encode(self, x):
        h = torch.relu(self.enc_fc1(x))
        h = torch.relu(self.enc_fc2(h))
        return self.fc_mu(h), self.fc_logvar(h)
    
    def reparameterize(self, mu, logvar):
        """Reparameterization trick: z = mu + eps * sigma."""
        if self.training:
            std = torch.exp(0.5 * logvar)
            eps = torch.randn_like(std)  # eps ~ N(0, I)
            return mu + eps * std
        return mu  # use mean at inference
    
    def decode(self, z):
        h = torch.relu(self.dec_fc1(z))
        h = torch.relu(self.dec_fc2(h))
        return self.dec_out(h)  # log-normalized expression (Gaussian output)
    
    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        x_hat = self.decode(z)
        return x_hat, mu, logvar

def elbo_loss(x, x_hat, mu, logvar, beta=1.0):
    """ELBO = reconstruction + beta * KL divergence."""
    reconstruction = nn.functional.mse_loss(x_hat, x, reduction='sum') / len(x)
    # KL: -0.5 * sum(1 + log_var - mu^2 - var)
    kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / len(x)
    return reconstruction + beta * kl, reconstruction.item(), kl.item()

# Dataset
class ScRNADataset(Dataset):
    def __init__(self, X): self.X = torch.tensor(X)
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i]

dataset = ScRNADataset(X_lognorm)
train_ds, val_ds = torch.utils.data.random_split(dataset, [500, 100])
train_ld = DataLoader(train_ds, batch_size=64, shuffle=True)
val_ld = DataLoader(val_ds, batch_size=128)

# Train
vae = VAE(input_dim=n_genes, latent_dim=10, hidden_dim=128).to(device)
optimizer_vae = optim.Adam(vae.parameters(), lr=1e-3)

train_losses, kl_hist, recon_hist = [], [], []
for epoch in range(80):
    vae.train()
    bl, kl_bl, re_bl = [], [], []
    for X_b in train_ld:
        X_b = X_b.to(device)
        optimizer_vae.zero_grad()
        x_hat, mu, logvar = vae(X_b)
        loss, recon, kl = elbo_loss(X_b, x_hat, mu, logvar, beta=1.0)
        loss.backward(); optimizer_vae.step()
        bl.append(loss.item()); kl_bl.append(kl); re_bl.append(recon)
    train_losses.append(np.mean(bl))
    kl_hist.append(np.mean(kl_bl))
    recon_hist.append(np.mean(re_bl))
    if (epoch+1) % 20 == 0:
        print(f'Epoch {epoch+1}: ELBO={train_losses[-1]:.4f} (recon={recon_hist[-1]:.4f}, KL={kl_hist[-1]:.4f})')

# Extract latent representations
vae.eval()
with torch.no_grad():
    X_tensor = torch.tensor(X_lognorm).to(device)
    mu_all, logvar_all = vae.encode(X_tensor)
    Z_latent = mu_all.cpu().numpy()  # use mean as point estimate
print(f'\nLatent space: {Z_latent.shape}')

In [ ]:
# Step 7 — Visualization
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
cmap = plt.cm.get_cmap('tab10', n_celltypes)

# Panel A: ELBO decomposition during training
ax = axes[0]
ax.plot(train_losses, 'k', lw=1.5, label='Total ELBO')
ax.plot(recon_hist, 'steelblue', lw=1.5, ls='--', label='Reconstruction')
ax.plot(kl_hist, 'tomato', lw=1.5, ls=':', label='KL divergence')
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.set_title('A. ELBO decomposition\n(recon drives down, KL regularizes)')
ax.legend(fontsize=8)

# Panel B: Latent space (first 2 dims), colored by cell type
from sklearn.decomposition import PCA
pca_lat = PCA(n_components=2)
Z_2d = pca_lat.fit_transform(Z_latent)
ax = axes[1]
for c in range(n_celltypes):
    mask = cell_types == c
    ax.scatter(Z_2d[mask, 0], Z_2d[mask, 1], c=[cmap(c)], s=20, alpha=0.7, label=f'CT{c}')
ax.set_xlabel('Latent PC1'); ax.set_ylabel('Latent PC2')
ax.set_title('B. VAE latent space (PCA)\ncolored by true cell type')
ax.legend(fontsize=8)

# Panel C: Latent space interpolation between two cell types
ax = axes[2]
# Find two cells of different types
cell0 = Z_latent[cell_types == 0][0]
cell3 = Z_latent[cell_types == 3][0]
alphas = np.linspace(0, 1, 10)
interp_z = np.array([cell0*(1-a) + cell3*a for a in alphas])
with torch.no_grad():
    interp_x = vae.decode(torch.tensor(interp_z, dtype=torch.float32).to(device)).cpu().numpy()
# Plot first 20 genes across interpolation
im = ax.imshow(interp_x[:, :20].T, cmap='RdBu_r', aspect='auto', vmin=-1, vmax=3)
plt.colorbar(im, ax=ax, shrink=0.7)
ax.set_xlabel('Interpolation step (CT0 → CT3)')
ax.set_ylabel('Gene')
ax.set_xticks(range(0, 10, 2))
ax.set_xticklabels([f'{a:.1f}' for a in alphas[::2]], fontsize=8)
ax.set_title('C. Latent interpolation\n(smooth gene expression transition)')

# Panel D: PCA vs. VAE separation (silhouette comparison)
from sklearn.metrics import silhouette_score
X_pca_2d = PCA(n_components=2).fit_transform(X_lognorm)
sil_pca = silhouette_score(X_pca_2d, cell_types)
sil_vae = silhouette_score(Z_latent, cell_types)
ax = axes[3]
ax.bar(['PCA (2 PCs)\nsilhouette', 'VAE (10D)\nsilhouette'],
         [sil_pca, sil_vae], color=['steelblue', 'tomato'], edgecolor='black', alpha=0.85, width=0.5)
ax.set_ylabel('Silhouette score\n(higher = better separation)')
ax.set_title('D. Cell type separation\n(VAE vs. linear PCA)')
ax.set_ylim(0, 1)
for i, v in enumerate([sil_pca, sil_vae]):
    ax.text(i, v+0.02, f'{v:.3f}', ha='center', fontsize=11, fontweight='bold')

plt.suptitle('Module 14 NB06: Autoencoders and VAEs', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('autoencoders_vaes.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 8 — Exercises

1. Set `beta=0` in the ELBO loss (remove the KL term). Train the model.
   What happens to the latent space? What does it look like when you sample from $\mathcal{N}(0,I)$?
2. Set `beta=10` (beta-VAE). How does the latent space change? Does it improve
   disentanglement (each dimension captures a different factor of variation)?
3. Derive the closed-form KL divergence $D_{KL}(\mathcal{N}(\mu,\sigma^2) \| \mathcal{N}(0,I))$
   from first principles using the definition $D_{KL}(P\|Q) = \int p \log(p/q)\, dx$.
4. Replace the Gaussian decoder with a Poisson decoder for count data. Implement
   `log_likelihood_poisson(x, lambda_hat)` where `lambda_hat = softplus(decoder_output)`.

---
## Step 10 — Quiz

1. Write the VAE ELBO. What are the two terms and what does each penalize?
2. Explain the reparameterization trick. Why is it necessary?
3. Why does the VAE produce a smoother, more useful latent space than a
   deterministic autoencoder for generation?
4. What does scVI use the VAE for? Why is negative binomial likelihood appropriate
   for scRNA-seq data?
5. What is the prior $p(z)$ in a standard VAE? What does it imply about the
   expected distribution of latent codes?

---
## Step 12 — Reflection

> *[In one paragraph: explain why the ELBO is a lower bound on the log-likelihood
> $\log p(x)$. What is the gap between the ELBO and the true log-likelihood,
> and when is it zero?]*

---
*Next: `07_graph_neural_networks.ipynb`*